In [ ]:
import src.utils as ut
from src.data import load_dataset_and_make_dataloaders
from src.sigma import build_sigma_schedule
from src.common import c_funcs
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import io
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
wd = Path("/storage/homefs/ds21n601/diffusion_project_DL/runs/20251210_155344_fm_cond")

_, info = load_dataset_and_make_dataloaders(dataset_name="FashionMNIST", root_dir="../data", batch_size=32)
chkp = ut.find_latest_checkpoint(wd / "checkpoints")
model = ut.build_model_for_sampling(wd / "config.yaml", device, info)
model = ut.load_ckpt_into_model(model, chkp, device)

In [ ]:
channels = info.image_channels
H = info.image_size
sigma_data = float(info.sigma_data)
n_images = 4
nfe = 64

imgs = []

sigmas = build_sigma_schedule(nfe)
sigmas = sigmas.to(device)

# sigmas: 1D tensor [sigma_0, sigma_1, ..., sigma_T] (assumed decreasing)
x = torch.randn(n_images, channels, H, H, device=device) * sigmas[0].to(device)
labels_uncond = torch.full((n_images,), -1, dtype=torch.long, device=device)

for i, sigma in enumerate(sigmas):
    sigma = sigma.to(device)
    sigma_next = (
        sigmas[i + 1].to(device)
        if i + 1 < len(sigmas)
        else torch.tensor(0.0, device=device)
    )

    sigma_b = sigma.repeat(n_images)
    c_in, c_out, c_skip, c_noise = c_funcs(sigma_b, sigma_data)
    cin_x = c_in.view(-1, 1, 1, 1) * x

    with torch.no_grad():
        pred = model(cin_x, c_noise.to(device), labels_uncond)

    x_denoised = c_skip.view(-1, 1, 1, 1) * x + c_out.view(-1, 1, 1, 1) * pred
    d = (x - x_denoised) / sigma.view(1, 1, 1, 1)
    x = x + d * (sigma_next - sigma).view(1, 1, 1, 1)
    imgs.append(ut.to_unit_range(x).cpu())

imgs = torch.stack(imgs, axis=0).squeeze()

In [ ]:
imgs.shape

In [ ]:
frames = []

for tt in range(imgs.shape[0]):

    fig, axx = plt.subplots(imgs.shape[1], 1, figsize=(1.5, 7))
    for ii, ax in enumerate(axx):
        ax.imshow(imgs[tt, ii], interpolation="none", cmap="grey")
        ax.set_axis_off()
    fig.suptitle(f"t = {tt}\nsigma = {sigmas[tt]:.3f}")

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)
    buf.seek(0)

    frame = Image.open(buf)
    frames.append(frame)

frames[0].save(
    'animation.gif',
    save_all=True,
    append_images=frames[1:],
    duration=120,
    loop=0
)